In [ ]:
import pandas as pd

# =========================
# LOAD STRUCTURED DATA
# =========================
sales = pd.read_csv("lap_sales.csv")

print(sales.columns)
print("Rows:", len(sales))

print(sales.head())

Index(['laptop_ID', 'Price_euros', 'Company', 'TypeName', 'Inches',
       'Resolution', 'Screen.Type', 'Cpu.Vendor', 'Cpu.Series',
       'Cpu.Speed..GHz.', 'Ram..GB.', 'Storage.Type', 'Memory..GB.',
       'Gpu.Vendor', 'OpSys', 'Weight..Kg.', 'average_rating', 'rating_count'],
      dtype='object')
Rows: 786
   laptop_ID  Price_euros Company   TypeName  Inches Resolution  Screen.Type  \
0          2       898.94   Apple  Ultrabook    13.3         HD  Unspecified   
1          3       575.00      HP   Notebook    15.6        FHD  Unspecified   
2          4      2537.45   Apple  Ultrabook    15.4        QHD          IPS   
3          5      1803.60   Apple  Ultrabook    13.3        QHD          IPS   
4          6       400.00    Acer   Notebook    15.6         HD  Unspecified   

  Cpu.Vendor Cpu.Series  Cpu.Speed..GHz.  Ram..GB. Storage.Type  Memory..GB.  \
0      Intel         i5              1.8         8          SSD          128   
1      Intel         i5              2.5      

In [ ]:
sales = sales.rename(columns={
    "Price_euros": "price",
    "Ram..GB.": "ram",
    "Memory..GB.": "storage",
    "Weight..Kg.": "weight",
    "rating_count": "demand"
})

# ensure numeric
sales["ram"] = pd.to_numeric(sales["ram"], errors="coerce")
sales["storage"] = pd.to_numeric(sales["storage"], errors="coerce")
sales["price"] = pd.to_numeric(sales["price"], errors="coerce")
sales["weight"] = pd.to_numeric(sales["weight"], errors="coerce")

sales = sales.dropna(subset=["price","ram","storage"])

print(sales.columns)

Index(['laptop_ID', 'price', 'Company', 'TypeName', 'Inches', 'Resolution',
       'Screen.Type', 'Cpu.Vendor', 'Cpu.Series', 'Cpu.Speed..GHz.', 'ram',
       'Storage.Type', 'storage', 'Gpu.Vendor', 'OpSys', 'weight',
       'average_rating', 'demand'],
      dtype='object')


In [ ]:

# HELPER FUNCTION

def recommend_config(df, filters, segment_name):

    temp = df.copy()

    for col, rule in filters.items():

        # categorical
        if isinstance(rule, list):
            temp = temp[temp[col].isin(rule)]

        # range
        elif isinstance(rule, tuple):
            low, high = rule
            temp = temp[(temp[col] >= low) & (temp[col] <= high)]

        # minimum
        else:
            temp = temp[temp[col] >= rule]

    if len(temp) == 0:
        print(f"\n===== {segment_name} =====")
        print("No matching configs")
        return

    temp = temp.sort_values("demand", ascending=False)
    top = temp.head(5)

    print(f"\n===== {segment_name} =====")
    print("Candidates:", len(temp))
    print("Avg price:", round(temp["price"].mean(), 2))
    print("Median price:", round(temp["price"].median(), 2))
    print(top[["Company","TypeName","ram","storage","Inches","weight","Cpu.Series","price","demand"]])

In [ ]:
# basic users - seg 1
recommend_config(
    sales,
    filters={
        "ram": (8, 8),
        "storage": (256, 512),
        "Inches": (12, 14),
        "weight": (0, 1.6)
    },
    segment_name="Segment 1 – Basic / Portable"
)


===== Segment 1 – Basic / Portable =====
Candidates: 109
Avg price: 1448.92
Median price: 1399.0
       Company   TypeName  ram  storage  Inches  weight Cpu.Series    price  \
25       Apple  Ultrabook    8      256    13.3   1.350         i5   998.00   
290  Microsoft  Ultrabook    8      256    13.5   1.250         i7  1799.00   
420  Microsoft  Ultrabook    8      256    13.5   1.252         i7  1867.85   
34       Apple  Ultrabook    8      256    13.3   1.370         i5  1419.00   
6        Apple  Ultrabook    8      256    13.3   1.340         i5  1158.70   

     demand  
25     2385  
290    1161  
420     937  
34      719  
6       599  


In [ ]:
# premium users - seg 2
recommend_config(
    sales,
    filters={
        "ram": (16, 32),
        "storage": (512, 1024),
        "Inches": (13, 15)
    },
    segment_name="Segment 2 – Premium"
)


===== Segment 2 – Premium =====
Candidates: 26
Avg price: 2179.61
Median price: 2074.0
       Company            TypeName  ram  storage  Inches  weight Cpu.Series  \
285  Microsoft           Ultrabook   16      512    13.5    1.25         i7   
392       Asus           Ultrabook   16      512    14.0    1.10         i7   
754      Razer              Gaming   16     1024    14.0    1.95         i7   
488      Razer              Gaming   16      512    14.0    1.95         i7   
207         HP  2 in 1 Convertible   16     1024    13.3    1.29         i7   

      price  demand  
285  2589.0    1159  
392  1900.0     111  
754  3499.0      95  
488  2899.0      95  
207  2449.0      95  


In [ ]:
# seg 3 - power users
recommend_config(
    sales,
    filters={
        "ram": (16, 32),
        "storage": (512, 2048),
        "Inches": (15, 18)
    },
    segment_name="Segment 3 – Power"
)


===== Segment 3 – Power =====
Candidates: 40
Avg price: 2156.55
Median price: 1899.0
    Company            TypeName  ram  storage  Inches  weight Cpu.Series  \
523    Asus              Gaming   16      512    17.3    2.73         i7   
110    Dell              Gaming   16      512    15.6    2.56         i7   
556     MSI              Gaming   16      512    17.3    2.43         i7   
718    Dell  2 in 1 Convertible   16      512    15.6    2.09         i7   
712  Lenovo              Gaming   16      512    15.6    3.31         i7   

       price  demand  
523  1799.00     659  
110  1249.26     307  
556  2649.00     228  
718  1179.00     194  
712  1305.00     148  


In [ ]:
# seg 4 - after sales support and reliability users
recommend_config(
    sales,
    filters={
        "ram": (8, 16),
        "storage": (256, 512),
        "price": (0, 1500)
    },
    segment_name="Segment 4 – Reliability"
)


===== Segment 4 – Reliability =====
Candidates: 202
Avg price: 1109.02
Median price: 1126.5
    Company   TypeName  ram  storage  Inches  weight Cpu.Series   price  \
25    Apple  Ultrabook    8      256    13.3    1.35         i5   998.0   
34    Apple  Ultrabook    8      256    13.3    1.37         i5  1419.0   
464    Dell   Notebook    8      256    15.6    2.00         i3   665.0   
6     Apple  Ultrabook    8      256    13.3    1.34         i5  1158.7   
160    Acer     Gaming    8      256    15.6    2.50         i5   846.0   

     demand  
25     2385  
34      719  
464     610  
6       599  
160     579  
